In [1]:
# Cell 1: Imports and Environment Setup
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Use first GPU

import json
import math
import time
import random
import logging
import warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Any
from collections import Counter, defaultdict
from datetime import datetime

import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    SiglipImageProcessor,
    SiglipVisionModel,
    get_cosine_schedule_with_warmup,
)

# Suppress warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print("✓ Imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

✓ Imports successful
PyTorch version: 2.9.1+cu128
CUDA available: True
GPU: NVIDIA H800 PCIe
GPU Memory: 79.11 GB


In [2]:
# Cell 2: Configuration for Non-VLM Baseline

@dataclass
class BaselineConfig:
    """
    Configuration for Non-VLM Baseline Model.
    
    This baseline uses SigLIP2 visual features + temporal model + classifier head
    to compare against the VLM approach.
    """
    
    # Data paths
    train_data: str = "/mnt/share/ali/VLM_Project/hospital_data/train.cleaned.json"
    val_data: Optional[str] = None
    test_data: str = "/mnt/share/ali/VLM_Project/hospital_data/test.cleaned.json"
    output_dir: str = "/mnt/share/ali/VLM_Project/checkpoints_baseline"
    
    # Model paths
    vision_model_path: str = "/mnt/share/ali/VLMs/hf_cache/hub/siglip2-model/"
    
    # Tasks
    target_tasks: List[str] = field(default_factory=lambda: [
        "step_classification",
        "stage_classification"
    ])
    
    # Training parameters
    epochs: int = 20
    batch_size: int = 4  # Can use larger batch since no LLM
    grad_accum: int = 4
    learning_rate: float = 1e-4
    weight_decay: float = 0.01
    warmup_ratio: float = 0.03
    max_grad_norm: float = 1.0
    
    # Visual settings
    num_frames: int = 8
    freeze_vision: bool = True  # Freeze SigLIP2 encoder
    
    # Temporal model settings
    temporal_model: str = "transformer"  # Options: "transformer", "lstm", "gru", "avg_pool"
    hidden_dim: int = 512
    num_layers: int = 4
    num_heads: int = 8
    dropout: float = 0.1
    
    # Data split
    val_split_ratio: float = 0.1
    val_split_seed: int = 42
    
    # Logging & checkpointing
    log_every_steps: int = 10
    eval_every_steps: int = 500
    save_every_steps: int = 500
    eval_samples: int = 100
    
    # Early stopping
    early_stopping: bool = True
    early_stopping_patience: int = 5
    early_stopping_min_delta: float = 0.001
    
    # Hardware
    num_workers: int = 4
    seed: int = 42


def setup_logging(output_dir: Optional[Path] = None):
    """Setup logging configuration."""
    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )
    
    logger = logging.getLogger("baseline")
    logger.setLevel(logging.INFO)
    logger.handlers = []
    
    # Console handler
    console = logging.StreamHandler()
    console.setFormatter(formatter)
    logger.addHandler(console)
    
    # File handler
    if output_dir:
        output_dir.mkdir(parents=True, exist_ok=True)
        file_handler = logging.FileHandler(output_dir / "training.log")
        file_handler.setFormatter(formatter)
        logger.addHandler(file_handler)
    
    return logger


def set_seed(seed: int):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# Create config and logger
config = BaselineConfig()
logger = setup_logging()

set_seed(config.seed)

logger.info("=" * 70)
logger.info("NON-VLM BASELINE MODEL")
logger.info("=" * 70)
logger.info(f"Temporal model: {config.temporal_model}")
logger.info(f"Hidden dim: {config.hidden_dim}")
logger.info(f"Num layers: {config.num_layers}")
logger.info(f"Tasks: {config.target_tasks}")
logger.info("=" * 70)

print("\n✓ Configuration created")
print(f"Output directory: {config.output_dir}")
print(f"Temporal model: {config.temporal_model}")

2026-01-18 01:34:52 | INFO     | ======================================================================
2026-01-18 01:34:52 | INFO     | NON-VLM BASELINE MODEL
2026-01-18 01:34:52 | INFO     | ======================================================================
2026-01-18 01:34:52 | INFO     | Temporal model: transformer
2026-01-18 01:34:52 | INFO     | Hidden dim: 512
2026-01-18 01:34:52 | INFO     | Num layers: 4
2026-01-18 01:34:52 | INFO     | Tasks: ['step_classification', 'stage_classification']
2026-01-18 01:34:52 | INFO     | ======================================================================



✓ Configuration created
Output directory: /mnt/share/ali/VLM_Project/checkpoints_baseline
Temporal model: transformer


In [4]:
# Cell 3: Data Loading and Analysis (Subject-based split)

class DatasetAnalyzer:
    """Analyze and display dataset statistics."""
    
    def __init__(self, samples: List[Dict], name: str = "Dataset"):
        self.samples = samples
        self.name = name
        self.stats = self._compute_stats()
    
    def _compute_stats(self) -> Dict[str, Any]:
        stats = {
            "total_samples": len(self.samples),
            "task_distribution": Counter(),
            "subject_distribution": Counter(),
            "frames_per_sample": [],
            "unique_labels": {},
        }
        
        for sample in self.samples:
            task = sample.get("main_tag", "unknown")
            stats["task_distribution"][task] += 1
            
            # Extract subject from metadata
            subject = sample.get("meta", {}).get("subject", "unknown")
            stats["subject_distribution"][subject] += 1
            
            frames = sample.get("frames", [])
            stats["frames_per_sample"].append(len(frames))
            
            # Extract label from conversations
            convs = sample.get("conversations", [])
            if len(convs) >= 2:
                label = convs[1].get("value", "").strip()
                if task not in stats["unique_labels"]:
                    stats["unique_labels"][task] = set()
                stats["unique_labels"][task].add(label)
        
        return stats
    
    def print_report(self):
        logger.info("=" * 70)
        logger.info(f"DATASET: {self.name}")
        logger.info("=" * 70)
        logger.info(f"Total samples: {self.stats['total_samples']}")
        
        logger.info("\nTask Distribution:")
        for task, count in sorted(self.stats['task_distribution'].items()):
            pct = 100 * count / self.stats['total_samples']
            num_classes = len(self.stats['unique_labels'].get(task, set()))
            logger.info(f"  {task:25s}: {count:6d} ({pct:5.1f}%) - {num_classes} classes")
        
        logger.info("\nSubject Distribution:")
        for subject, count in sorted(self.stats['subject_distribution'].items()):
            pct = 100 * count / self.stats['total_samples']
            logger.info(f"  {subject:25s}: {count:6d} ({pct:5.1f}%)")
        
        if self.stats['frames_per_sample']:
            frames = self.stats['frames_per_sample']
            logger.info(f"\nFrames per sample: min={min(frames)}, max={max(frames)}, mean={np.mean(frames):.1f}")
        
        logger.info("=" * 70)
    
    def get_label_mapping(self) -> Dict[str, Dict[str, int]]:
        """Create label to index mapping for each task."""
        label_maps = {}
        for task, labels in self.stats['unique_labels'].items():
            sorted_labels = sorted(list(labels))
            label_maps[task] = {label: idx for idx, label in enumerate(sorted_labels)}
        return label_maps


def load_and_split_data(config: BaselineConfig) -> Tuple[List[Dict], List[Dict], List[Dict], Dict]:
    """Load and split data by subject, return label mappings."""
    logger.info("=" * 70)
    logger.info("LOADING DATA")
    logger.info("=" * 70)
    
    # Load training data
    logger.info(f"Loading training data: {config.train_data}")
    with open(config.train_data, "r", encoding="utf-8") as f:
        all_train = json.load(f)
    
    # Filter by tasks
    if config.target_tasks:
        train_samples = [s for s in all_train if s.get("main_tag") in config.target_tasks]
        logger.info(f"Filtered to {len(train_samples)} samples for tasks: {config.target_tasks}")
    else:
        train_samples = all_train
    
    # Create validation split BY SUBJECT
    if config.val_data and os.path.exists(config.val_data):
        logger.info(f"Loading validation data: {config.val_data}")
        with open(config.val_data, "r", encoding="utf-8") as f:
            all_val = json.load(f)
        if config.target_tasks:
            val_samples = [s for s in all_val if s.get("main_tag") in config.target_tasks]
        else:
            val_samples = all_val
        logger.info(f"Loaded {len(val_samples)} validation samples from file")
    else:
        logger.info(f"Creating SUBJECT-BASED validation split from training data")
        
        # Group samples by subject
        subject_samples = defaultdict(list)
        for sample in train_samples:
            subject = sample.get("meta", {}).get("subject", "unknown")
            subject_samples[subject].append(sample)
        
        all_subjects = sorted(subject_samples.keys())
        logger.info(f"Found {len(all_subjects)} unique subjects: {all_subjects}")
        
        # Calculate samples per subject
        for subj in all_subjects:
            logger.info(f"  {subj}: {len(subject_samples[subj])} samples")
        
        # Select validation subjects (roughly config.val_split_ratio of data)
        rng = random.Random(config.val_split_seed)
        shuffled_subjects = all_subjects.copy()
        rng.shuffle(shuffled_subjects)
        
        # Greedily select subjects until we reach target ratio
        val_subjects = []
        val_count = 0
        target_val_count = int(len(train_samples) * config.val_split_ratio)
        
        for subj in shuffled_subjects:
            if val_count < target_val_count:
                val_subjects.append(subj)
                val_count += len(subject_samples[subj])
            else:
                break
        
        # If no subjects selected, take at least one
        if not val_subjects:
            val_subjects = [shuffled_subjects[0]]
            val_count = len(subject_samples[val_subjects[0]])
        
        # Split samples
        val_samples = []
        train_samples_new = []
        
        for sample in train_samples:
            subject = sample.get("meta", {}).get("subject", "unknown")
            if subject in val_subjects:
                val_samples.append(sample)
            else:
                train_samples_new.append(sample)
        
        train_samples = train_samples_new
        
        logger.info(f"\nSubject-based split:")
        logger.info(f"  - Validation subjects: {sorted(val_subjects)}")
        logger.info(f"  - Training subjects: {sorted(set(all_subjects) - set(val_subjects))}")
        logger.info(f"  - Training samples: {len(train_samples)}")
        logger.info(f"  - Validation samples: {len(val_samples)} ({len(val_samples)/len(train_samples+val_samples)*100:.1f}%)")
    
    # Load test data
    test_samples = []
    if config.test_data and os.path.exists(config.test_data):
        logger.info(f"Loading test data: {config.test_data}")
        with open(config.test_data, "r", encoding="utf-8") as f:
            all_test = json.load(f)
        if config.target_tasks:
            test_samples = [s for s in all_test if s.get("main_tag") in config.target_tasks]
        else:
            test_samples = all_test
        logger.info(f"Loaded {len(test_samples)} test samples")
    
    # Verify no subject overlap between train and val
    train_subjects = {s.get("meta", {}).get("subject", "unknown") for s in train_samples}
    val_subjects = {s.get("meta", {}).get("subject", "unknown") for s in val_samples}
    test_subjects = {s.get("meta", {}).get("subject", "unknown") for s in test_samples}
    
    overlap_train_val = train_subjects & val_subjects
    overlap_train_test = train_subjects & test_subjects
    overlap_val_test = val_subjects & test_subjects
    
    if overlap_train_val:
        logger.error(f"⚠️ SUBJECT LEAKAGE: {overlap_train_val} in both train and val!")
    if overlap_train_test:
        logger.error(f"⚠️ SUBJECT LEAKAGE: {overlap_train_test} in both train and test!")
    if overlap_val_test:
        logger.error(f"⚠️ SUBJECT LEAKAGE: {overlap_val_test} in both val and test!")
    
    if not (overlap_train_val or overlap_train_test or overlap_val_test):
        logger.info("✓ No subject leakage detected")
    
    # Analyze datasets
    train_analyzer = DatasetAnalyzer(train_samples, "TRAINING")
    train_analyzer.print_report()
    
    val_analyzer = DatasetAnalyzer(val_samples, "VALIDATION")
    val_analyzer.print_report()
    
    if test_samples:
        test_analyzer = DatasetAnalyzer(test_samples, "TEST")
        test_analyzer.print_report()
    
    # Get label mappings from training data
    label_mappings = train_analyzer.get_label_mapping()
    logger.info("\nLabel Mappings:")
    for task, mapping in label_mappings.items():
        logger.info(f"  {task}: {len(mapping)} classes")
        for label, idx in sorted(mapping.items(), key=lambda x: x[1]):
            logger.info(f"    {idx}: {label}")
    
    return train_samples, val_samples, test_samples, label_mappings


# Load data
train_samples, val_samples, test_samples, label_mappings = load_and_split_data(config)

print("\n✓ Data loaded successfully")
print(f"Training samples: {len(train_samples)}")
print(f"Validation samples: {len(val_samples)}")
print(f"Test samples: {len(test_samples)}")
print(f"Label mappings: {list(label_mappings.keys())}")

2026-01-18 01:37:54 | INFO     | ======================================================================
2026-01-18 01:37:54 | INFO     | LOADING DATA
2026-01-18 01:37:54 | INFO     | ======================================================================
2026-01-18 01:37:54 | INFO     | Loading training data: /mnt/share/ali/VLM_Project/hospital_data/train.cleaned.json
2026-01-18 01:37:54 | INFO     | Filtered to 758 samples for tasks: ['step_classification', 'stage_classification']
2026-01-18 01:37:54 | INFO     | Creating SUBJECT-BASED validation split from training data
2026-01-18 01:37:54 | INFO     | Found 33 unique subjects: ['万佳', '何丽娟', '冯梦帆', '刘映辉', '吴俊东', '周泽华', '唐婧怡', '廖雨冰', '张泽楠', '李万春', '李泽伟', '李浪', '李湘赞', '李源', '林翥鸿', '梁毅', '毛欣蕾', '滕发满', '王帅威', '王绍禹', '罗芷珊', '苏钰淇', '范博甲', '蔡建辉', '蔡毓钦', '覃妮香', '谢佳霖', '邓椅汶', '陈思泳', '陈赞之', '陈飞瑶', '韩祥龙', '黄晓漫']
2026-01-18 01:37:54 | INFO     |   万佳: 20 samples
2026-01-18 01:37:54 | INFO     |   何丽娟: 24 samples
2026-01-18 01:37:54 | INFO     |  


✓ Data loaded successfully
Training samples: 666
Validation samples: 92
Test samples: 224
Label mappings: ['step_classification', 'stage_classification']


In [5]:
# Cell 4: Dataset Class for Baseline Model

class BaselineDataset(Dataset):
    """Dataset for baseline model (no text, just vision + labels)."""
    
    def __init__(
        self,
        samples: List[Dict],
        image_processor: SiglipImageProcessor,
        label_mappings: Dict[str, Dict[str, int]],
        num_frames: int = 8,
    ):
        self.samples = self._validate_samples(samples)
        self.image_processor = image_processor
        self.label_mappings = label_mappings
        self.num_frames = num_frames
    
    def _validate_samples(self, samples: List[Dict]) -> List[Dict]:
        """Filter out invalid samples."""
        valid = []
        for s in samples:
            has_frames = "frames" in s and len(s["frames"]) > 0
            has_convs = "conversations" in s and len(s["conversations"]) >= 2
            has_task = "main_tag" in s
            if has_frames and has_convs and has_task:
                valid.append(s)
        
        if len(valid) < len(samples):
            logger.warning(f"Filtered {len(samples) - len(valid)} invalid samples")
        
        return valid
    
    def _load_frames(self, frame_paths: List[str]) -> List[Image.Image]:
        """Load and pad frames to num_frames."""
        images = []
        
        for path in frame_paths[:self.num_frames]:
            path = path.replace("\\", "/")
            try:
                if os.path.exists(path):
                    img = Image.open(path).convert("RGB")
                else:
                    img = Image.new("RGB", (384, 384), color=(128, 128, 128))
            except Exception as e:
                img = Image.new("RGB", (384, 384), color=(128, 128, 128))
            images.append(img)
        
        # Pad if needed
        while len(images) < self.num_frames:
            images.append(images[-1].copy() if images else Image.new("RGB", (384, 384)))
        
        return images[:self.num_frames]
    
    def __len__(self) -> int:
        return len(self.samples)
    
    def __getitem__(self, idx: int) -> Dict[str, Any]:
        sample = self.samples[idx]
        
        # Load frames
        frames = self._load_frames(sample.get("frames", []))
        processed = self.image_processor(images=frames, return_tensors="pt")
        pixel_values = processed["pixel_values"]  # [N, C, H, W]
        
        # Get label
        task = sample.get("main_tag", "unknown")
        convs = sample.get("conversations", [])
        label_text = convs[1]["value"] if len(convs) > 1 else ""
        
        # Map label to index
        if task in self.label_mappings:
            label_idx = self.label_mappings[task].get(label_text.strip(), -1)
        else:
            label_idx = -1
        
        return {
            "pixel_values": pixel_values,  # [num_frames, C, H, W]
            "label": label_idx,
            "task": task,
            "sample_id": sample.get("id", f"sample_{idx}"),
        }


def collate_fn(batch: List[Dict]) -> Dict[str, Any]:
    """Collate batch samples."""
    pixel_values = torch.stack([b["pixel_values"] for b in batch])  # [B, N, C, H, W]
    labels = torch.tensor([b["label"] for b in batch], dtype=torch.long)
    
    return {
        "pixel_values": pixel_values,
        "labels": labels,
        "tasks": [b["task"] for b in batch],
        "sample_ids": [b["sample_id"] for b in batch],
    }


# Create datasets
logger.info("=" * 70)
logger.info("CREATING DATASETS")
logger.info("=" * 70)

# Load image processor
logger.info(f"Loading SigLIP image processor: {config.vision_model_path}")
image_processor = SiglipImageProcessor.from_pretrained(config.vision_model_path)

train_dataset = BaselineDataset(
    samples=train_samples,
    image_processor=image_processor,
    label_mappings=label_mappings,
    num_frames=config.num_frames,
)

val_dataset = BaselineDataset(
    samples=val_samples,
    image_processor=image_processor,
    label_mappings=label_mappings,
    num_frames=config.num_frames,
)

test_dataset = BaselineDataset(
    samples=test_samples,
    image_processor=image_processor,
    label_mappings=label_mappings,
    num_frames=config.num_frames,
)

logger.info(f"Train dataset: {len(train_dataset)} samples")
logger.info(f"Val dataset: {len(val_dataset)} samples")
logger.info(f"Test dataset: {len(test_dataset)} samples")

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    collate_fn=collate_fn,
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    collate_fn=collate_fn,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    collate_fn=collate_fn,
    pin_memory=True,
)

logger.info(f"Train batches: {len(train_loader)}")
logger.info(f"Val batches: {len(val_loader)}")
logger.info(f"Test batches: {len(test_loader)}")

# Test loading a batch
logger.info("\nTesting data loading...")
test_batch = next(iter(train_loader))
logger.info(f"Batch shapes:")
logger.info(f"  pixel_values: {test_batch['pixel_values'].shape}")
logger.info(f"  labels: {test_batch['labels'].shape}")
logger.info(f"  tasks: {len(test_batch['tasks'])}")
logger.info("=" * 70)

print("\n✓ Datasets created successfully")
print(f"Train batches: {len(train_loader)}")
print(f"Batch size: {config.batch_size}")
print(f"Sample batch - pixel_values shape: {test_batch['pixel_values'].shape}")
print(f"Sample batch - labels shape: {test_batch['labels'].shape}")

2026-01-18 01:38:52 | INFO     | ======================================================================
2026-01-18 01:38:52 | INFO     | CREATING DATASETS
2026-01-18 01:38:52 | INFO     | ======================================================================
2026-01-18 01:38:52 | INFO     | Loading SigLIP image processor: /mnt/share/ali/VLMs/hf_cache/hub/siglip2-model/
2026-01-18 01:38:52 | INFO     | Train dataset: 666 samples
2026-01-18 01:38:52 | INFO     | Val dataset: 92 samples
2026-01-18 01:38:52 | INFO     | Test dataset: 224 samples
2026-01-18 01:38:52 | INFO     | Train batches: 166
2026-01-18 01:38:52 | INFO     | Val batches: 23
2026-01-18 01:38:52 | INFO     | Test batches: 56
2026-01-18 01:38:52 | INFO     | 
Testing data loading...
2026-01-18 01:38:54 | INFO     | Batch shapes:
2026-01-18 01:38:54 | INFO     |   pixel_values: torch.Size([4, 8, 3, 384, 384])
2026-01-18 01:38:54 | INFO     |   labels: torch.Size([4])
2026-01-18 01:38:54 | INFO     |   tasks: 4
2026-01-18 0


✓ Datasets created successfully
Train batches: 166
Batch size: 4
Sample batch - pixel_values shape: torch.Size([4, 8, 3, 384, 384])
Sample batch - labels shape: torch.Size([4])


In [6]:
# Cell 5: Baseline Model Architecture

class TemporalTransformer(nn.Module):
    """Transformer-based temporal model for frame sequences."""
    
    def __init__(self, hidden_dim: int, num_layers: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        
        self.pos_embed = nn.Parameter(torch.zeros(1, 8, hidden_dim))  # max 8 frames
        nn.init.normal_(self.pos_embed, std=0.02)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
        )
        
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(hidden_dim)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: [B, N_frames, D]
        Returns:
            pooled: [B, D]
        """
        # Add positional embeddings
        x = x + self.pos_embed[:, :x.size(1), :]
        
        # Transformer encoding
        x = self.transformer(x)  # [B, N_frames, D]
        x = self.norm(x)
        
        # Temporal pooling (mean over frames)
        pooled = x.mean(dim=1)  # [B, D]
        
        return pooled


class TemporalLSTM(nn.Module):
    """LSTM-based temporal model for frame sequences."""
    
    def __init__(self, hidden_dim: int, num_layers: int, dropout: float = 0.1):
        super().__init__()
        
        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True,
            bidirectional=True,
        )
        
        self.projection = nn.Linear(hidden_dim * 2, hidden_dim)  # bidirectional
        self.norm = nn.LayerNorm(hidden_dim)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: [B, N_frames, D]
        Returns:
            pooled: [B, D]
        """
        # LSTM encoding
        output, (hidden, cell) = self.lstm(x)  # output: [B, N_frames, D*2]
        
        # Use last hidden state (concatenate both directions)
        # hidden: [num_layers*2, B, D] -> take last layer
        last_hidden = hidden[-2:, :, :]  # [2, B, D] (forward and backward)
        last_hidden = last_hidden.transpose(0, 1).contiguous()  # [B, 2, D]
        last_hidden = last_hidden.view(last_hidden.size(0), -1)  # [B, D*2]
        
        # Project back to hidden_dim
        pooled = self.projection(last_hidden)  # [B, D]
        pooled = self.norm(pooled)
        
        return pooled


class TemporalGRU(nn.Module):
    """GRU-based temporal model for frame sequences."""
    
    def __init__(self, hidden_dim: int, num_layers: int, dropout: float = 0.1):
        super().__init__()
        
        self.gru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True,
            bidirectional=True,
        )
        
        self.projection = nn.Linear(hidden_dim * 2, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: [B, N_frames, D]
        Returns:
            pooled: [B, D]
        """
        output, hidden = self.gru(x)  # hidden: [num_layers*2, B, D]
        
        # Use last hidden state
        last_hidden = hidden[-2:, :, :]  # [2, B, D]
        last_hidden = last_hidden.transpose(0, 1).contiguous()  # [B, 2, D]
        last_hidden = last_hidden.view(last_hidden.size(0), -1)  # [B, D*2]
        
        pooled = self.projection(last_hidden)
        pooled = self.norm(pooled)
        
        return pooled


class TemporalAvgPool(nn.Module):
    """Simple average pooling over frames (baseline)."""
    
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.norm = nn.LayerNorm(hidden_dim)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: [B, N_frames, D]
        Returns:
            pooled: [B, D]
        """
        pooled = x.mean(dim=1)  # [B, D]
        pooled = self.norm(pooled)
        return pooled


class BaselineModel(nn.Module):
    """
    Baseline Model: SigLIP2 + Temporal Model + Classifier Head
    
    This serves as a comparison to the VLM approach to justify using
    language models and instruction format.
    """
    
    def __init__(
        self,
        vision_encoder: SiglipVisionModel,
        label_mappings: Dict[str, Dict[str, int]],
        config: BaselineConfig,
    ):
        super().__init__()
        
        self.vision_encoder = vision_encoder
        self.label_mappings = label_mappings
        self.config = config
        
        # Get dimensions
        self.vision_hidden = vision_encoder.config.hidden_size
        self.hidden_dim = config.hidden_dim
        
        logger.info(f"Vision hidden: {self.vision_hidden}")
        logger.info(f"Model hidden: {self.hidden_dim}")
        
        # Visual projection (reduce dimension if needed)
        if self.vision_hidden != self.hidden_dim:
            self.visual_proj = nn.Sequential(
                nn.Linear(self.vision_hidden, self.hidden_dim),
                nn.LayerNorm(self.hidden_dim),
                nn.GELU(),
            )
        else:
            self.visual_proj = nn.Identity()
        
        # Temporal model
        logger.info(f"Creating temporal model: {config.temporal_model}")
        if config.temporal_model == "transformer":
            self.temporal_model = TemporalTransformer(
                hidden_dim=self.hidden_dim,
                num_layers=config.num_layers,
                num_heads=config.num_heads,
                dropout=config.dropout,
            )
        elif config.temporal_model == "lstm":
            self.temporal_model = TemporalLSTM(
                hidden_dim=self.hidden_dim,
                num_layers=config.num_layers,
                dropout=config.dropout,
            )
        elif config.temporal_model == "gru":
            self.temporal_model = TemporalGRU(
                hidden_dim=self.hidden_dim,
                num_layers=config.num_layers,
                dropout=config.dropout,
            )
        elif config.temporal_model == "avg_pool":
            self.temporal_model = TemporalAvgPool(hidden_dim=self.hidden_dim)
        else:
            raise ValueError(f"Unknown temporal model: {config.temporal_model}")
        
        # Classification heads (one per task)
        self.classifiers = nn.ModuleDict()
        for task, label_map in label_mappings.items():
            num_classes = len(label_map)
            self.classifiers[task] = nn.Sequential(
                nn.Linear(self.hidden_dim, self.hidden_dim),
                nn.LayerNorm(self.hidden_dim),
                nn.GELU(),
                nn.Dropout(config.dropout),
                nn.Linear(self.hidden_dim, num_classes),
            )
            logger.info(f"Classifier for {task}: {num_classes} classes")
        
        # Freeze vision encoder if specified
        if config.freeze_vision:
            for p in self.vision_encoder.parameters():
                p.requires_grad = False
            self.vision_encoder.eval()
            logger.info("Vision encoder frozen")
    
    def encode_frames(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """
        Encode frames using vision encoder.
        
        Args:
            pixel_values: [B, N_frames, C, H, W]
        Returns:
            frame_features: [B, N_frames, D]
        """
        B, N, C, H, W = pixel_values.shape
        
        # Flatten: [B*N, C, H, W]
        flat = pixel_values.view(B * N, C, H, W)
        
        # Encode
        with torch.no_grad() if self.config.freeze_vision else torch.enable_grad():
            outputs = self.vision_encoder(pixel_values=flat)
            features = outputs.pooler_output  # [B*N, vision_hidden]
        
        # Reshape: [B, N, vision_hidden]
        frame_features = features.view(B, N, -1)
        
        # Project to model hidden dim
        frame_features = self.visual_proj(frame_features)  # [B, N, hidden_dim]
        
        return frame_features
    
    def forward(self, pixel_values: torch.Tensor, labels: torch.Tensor, tasks: List[str]) -> Dict[str, torch.Tensor]:
        """
        Forward pass with mixed-task batch.
        
        Args:
            pixel_values: [B, N_frames, C, H, W]
            labels: [B]
            tasks: List of task names (length B)
        Returns:
            dict with 'loss' and per-task losses
        """
        # Encode frames
        frame_features = self.encode_frames(pixel_values)  # [B, N_frames, hidden_dim]
        
        # Temporal modeling
        pooled_features = self.temporal_model(frame_features)  # [B, hidden_dim]
        
        # Classify per task
        losses = []
        task_losses = defaultdict(list)
        
        for i, task in enumerate(tasks):
            if task in self.classifiers:
                logits = self.classifiers[task](pooled_features[i:i+1])  # [1, num_classes]
                target = labels[i:i+1]  # [1]
                
                if target.item() >= 0:  # valid label
                    loss = F.cross_entropy(logits, target)
                    losses.append(loss)
                    task_losses[task].append(loss.item())
        
        # Average loss
        if losses:
            total_loss = torch.stack(losses).mean()
        else:
            total_loss = torch.tensor(0.0, device=pixel_values.device)
        
        return {
            "loss": total_loss,
            "task_losses": {k: np.mean(v) for k, v in task_losses.items()},
        }
    
    @torch.no_grad()
    def predict(self, pixel_values: torch.Tensor, task: str) -> torch.Tensor:
        """
        Predict class for a batch.
        
        Args:
            pixel_values: [B, N_frames, C, H, W]
            task: task name
        Returns:
            predictions: [B]
        """
        self.eval()
        
        # Encode frames
        frame_features = self.encode_frames(pixel_values)
        
        # Temporal modeling
        pooled_features = self.temporal_model(frame_features)
        
        # Classify
        if task in self.classifiers:
            logits = self.classifiers[task](pooled_features)
            predictions = logits.argmax(dim=-1)
        else:
            predictions = torch.zeros(pixel_values.size(0), dtype=torch.long, device=pixel_values.device)
        
        return predictions


# Count parameters
def count_parameters(model: nn.Module) -> Dict[str, int]:
    """Count trainable and total parameters."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = total - trainable
    
    return {
        "total": total,
        "trainable": trainable,
        "frozen": frozen,
    }


print("\n✓ Model architecture defined")
print("Available temporal models: transformer, lstm, gru, avg_pool")
print(f"Current config: {config.temporal_model}")


✓ Model architecture defined
Available temporal models: transformer, lstm, gru, avg_pool
Current config: transformer


In [7]:
# Cell 6: Load Vision Encoder and Create Baseline Model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

logger.info("=" * 70)
logger.info("LOADING VISION ENCODER")
logger.info("=" * 70)

# Load SigLIP vision encoder
logger.info(f"Loading SigLIP from: {config.vision_model_path}")
vision_encoder = SiglipVisionModel.from_pretrained(config.vision_model_path)
vision_encoder = vision_encoder.to(device)

logger.info(f"Vision encoder loaded: {vision_encoder.config.hidden_size} hidden dim")

# Create baseline model
logger.info("\n" + "=" * 70)
logger.info("CREATING BASELINE MODEL")
logger.info("=" * 70)

model = BaselineModel(
    vision_encoder=vision_encoder,
    label_mappings=label_mappings,
    config=config,
)

model = model.to(device)

# Count parameters
param_counts = count_parameters(model)

logger.info("\n" + "=" * 70)
logger.info("MODEL STATISTICS")
logger.info("=" * 70)
logger.info(f"Total parameters: {param_counts['total']:,}")
logger.info(f"Trainable parameters: {param_counts['trainable']:,}")
logger.info(f"Frozen parameters: {param_counts['frozen']:,}")
logger.info(f"Trainable %: {100 * param_counts['trainable'] / param_counts['total']:.2f}%")

# Count parameters by component
vision_params = sum(p.numel() for p in model.vision_encoder.parameters())
proj_params = sum(p.numel() for p in model.visual_proj.parameters())
temporal_params = sum(p.numel() for p in model.temporal_model.parameters())
classifier_params = sum(p.numel() for p in model.classifiers.parameters())

logger.info("\nParameters by component:")
logger.info(f"  Vision encoder: {vision_params:,}")
logger.info(f"  Visual projection: {proj_params:,}")
logger.info(f"  Temporal model: {temporal_params:,}")
logger.info(f"  Classifiers: {classifier_params:,}")

logger.info("\n" + "=" * 70)
logger.info("TESTING FORWARD PASS")
logger.info("=" * 70)

# Test forward pass
model.train()
test_batch = next(iter(train_loader))
test_pixel_values = test_batch["pixel_values"].to(device)
test_labels = test_batch["labels"].to(device)
test_tasks = test_batch["tasks"]

logger.info(f"Input shape: {test_pixel_values.shape}")
logger.info(f"Labels shape: {test_labels.shape}")
logger.info(f"Tasks: {test_tasks}")

# Forward pass
output = model(test_pixel_values, test_labels, test_tasks)

logger.info(f"\nOutput loss: {output['loss'].item():.4f}")
logger.info(f"Task losses: {output['task_losses']}")

# Test prediction
model.eval()
predictions = model.predict(test_pixel_values[:1], test_tasks[0])
logger.info(f"\nPrediction test: {predictions}")

logger.info("=" * 70)

# GPU memory info
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    logger.info(f"\nGPU Memory:")
    logger.info(f"  Allocated: {allocated:.2f} GB")
    logger.info(f"  Reserved: {reserved:.2f} GB")

print("\n✓ Model loaded and tested successfully")
print(f"Total parameters: {param_counts['total']:,}")
print(f"Trainable parameters: {param_counts['trainable']:,}")
print(f"Test forward pass - loss: {output['loss'].item():.4f}")

2026-01-18 01:40:31 | INFO     | ======================================================================
2026-01-18 01:40:31 | INFO     | LOADING VISION ENCODER
2026-01-18 01:40:31 | INFO     | ======================================================================
2026-01-18 01:40:31 | INFO     | Loading SigLIP from: /mnt/share/ali/VLMs/hf_cache/hub/siglip2-model/
2026-01-18 01:40:32 | INFO     | Vision encoder loaded: 1152 hidden dim
2026-01-18 01:40:32 | INFO     | 
2026-01-18 01:40:32 | INFO     | CREATING BASELINE MODEL
2026-01-18 01:40:32 | INFO     | ======================================================================
2026-01-18 01:40:32 | INFO     | Vision hidden: 1152
2026-01-18 01:40:32 | INFO     | Model hidden: 512
2026-01-18 01:40:32 | INFO     | Creating temporal model: transformer
2026-01-18 01:40:32 | INFO     | Classifier for step_classification: 13 classes
2026-01-18 01:40:32 | INFO     | Classifier for stage_classification: 3 classes
2026-01-18 01:40:32 | INFO     | 


✓ Model loaded and tested successfully
Total parameters: 441,967,184
Trainable parameters: 13,741,584
Test forward pass - loss: 2.2673


In [8]:
# Cell 7: Training and Evaluation Functions

@torch.no_grad()
def evaluate_model(
    model: BaselineModel,
    dataloader: DataLoader,
    device: torch.device,
    split_name: str = "validation"
) -> Dict[str, Any]:
    """Evaluate model on a dataset."""
    model.eval()
    
    all_predictions = []
    all_labels = []
    all_tasks = []
    
    total_loss = 0.0
    num_batches = 0
    
    for batch in tqdm(dataloader, desc=f"Evaluating {split_name}", leave=False):
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)
        tasks = batch["tasks"]
        
        # Get predictions for each sample
        for i in range(len(tasks)):
            task = tasks[i]
            pred = model.predict(pixel_values[i:i+1], task)
            
            all_predictions.append(pred.item())
            all_labels.append(labels[i].item())
            all_tasks.append(task)
        
        # Compute loss
        output = model(pixel_values, labels, tasks)
        total_loss += output["loss"].item()
        num_batches += 1
    
    # Compute metrics
    all_predictions = np.array(all_predictions)
    all_labels = np.array(all_labels)
    
    # Filter out invalid labels
    valid_mask = all_labels >= 0
    all_predictions = all_predictions[valid_mask]
    all_labels = all_labels[valid_mask]
    all_tasks = [t for i, t in enumerate(all_tasks) if valid_mask[i]]
    
    # Overall accuracy
    correct = (all_predictions == all_labels).sum()
    total = len(all_labels)
    accuracy = correct / total if total > 0 else 0.0
    
    # Per-task accuracy
    task_metrics = {}
    for task in set(all_tasks):
        task_mask = np.array([t == task for t in all_tasks])
        task_preds = all_predictions[task_mask]
        task_labels = all_labels[task_mask]
        
        task_correct = (task_preds == task_labels).sum()
        task_total = len(task_labels)
        task_acc = task_correct / task_total if task_total > 0 else 0.0
        
        task_metrics[task] = {
            "accuracy": task_acc,
            "correct": int(task_correct),
            "total": int(task_total),
        }
    
    avg_loss = total_loss / num_batches if num_batches > 0 else 0.0
    
    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "correct": int(correct),
        "total": int(total),
        "task_metrics": task_metrics,
        "split": split_name,
    }


def log_metrics(metrics: Dict[str, Any], logger, step: Optional[int] = None):
    """Log evaluation metrics."""
    prefix = f"[Step {step}] " if step is not None else ""
    split = metrics.get("split", "eval")
    
    logger.info(f"{prefix}{split.upper()} METRICS:")
    logger.info(f"  Loss: {metrics['loss']:.4f}")
    logger.info(f"  Accuracy: {metrics['accuracy']:.4f} ({metrics['correct']}/{metrics['total']})")
    
    if "task_metrics" in metrics:
        logger.info(f"  Per-task accuracy:")
        for task, task_metrics in metrics["task_metrics"].items():
            acc = task_metrics["accuracy"]
            correct = task_metrics["correct"]
            total = task_metrics["total"]
            logger.info(f"    {task:25s}: {acc:.4f} ({correct}/{total})")


def save_checkpoint(
    model: BaselineModel,
    optimizer,
    scheduler,
    epoch: int,
    step: int,
    metrics: Dict[str, Any],
    output_dir: Path,
    name: str = "checkpoint"
):
    """Save model checkpoint."""
    ckpt_dir = output_dir / name
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    
    # Save model state
    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "epoch": epoch,
        "step": step,
        "metrics": metrics,
    }, ckpt_dir / "model.pt")
    
    # Save config and label mappings
    with open(ckpt_dir / "config.json", "w") as f:
        json.dump({
            "config": {
                "temporal_model": config.temporal_model,
                "hidden_dim": config.hidden_dim,
                "num_layers": config.num_layers,
                "num_heads": config.num_heads,
                "dropout": config.dropout,
                "num_frames": config.num_frames,
            },
            "label_mappings": {k: {label: int(idx) for label, idx in v.items()} 
                              for k, v in label_mappings.items()},
            "metrics": metrics,
            "timestamp": datetime.now().isoformat(),
        }, f, indent=2)
    
    logger.info(f"Saved checkpoint: {ckpt_dir}")
    return ckpt_dir


def load_checkpoint(model: BaselineModel, checkpoint_path: Path, optimizer=None, scheduler=None):
    """Load model from checkpoint."""
    ckpt_file = checkpoint_path / "model.pt"
    
    if not ckpt_file.exists():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_file}")
    
    logger.info(f"Loading checkpoint: {ckpt_file}")
    checkpoint = torch.load(ckpt_file, map_location="cpu")
    
    model.load_state_dict(checkpoint["model_state_dict"])
    
    if optimizer is not None and "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    
    if scheduler is not None and "scheduler_state_dict" in checkpoint:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    
    epoch = checkpoint.get("epoch", 0)
    step = checkpoint.get("step", 0)
    metrics = checkpoint.get("metrics", {})
    
    logger.info(f"Loaded checkpoint from epoch {epoch}, step {step}")
    return epoch, step, metrics


def format_time(seconds: float) -> str:
    """Format seconds into human-readable string."""
    if seconds < 60:
        return f"{seconds:.1f}s"
    elif seconds < 3600:
        return f"{seconds/60:.1f}m"
    else:
        hours = int(seconds // 3600)
        minutes = int((seconds % 3600) // 60)
        return f"{hours}h {minutes}m"


print("\n✓ Training and evaluation functions defined")
print("Functions available:")
print("  - evaluate_model()")
print("  - log_metrics()")
print("  - save_checkpoint()")
print("  - load_checkpoint()")


✓ Training and evaluation functions defined
Functions available:
  - evaluate_model()
  - log_metrics()
  - save_checkpoint()
  - load_checkpoint()


In [9]:
# Cell 8: Main Training Loop

def train_baseline_model(
    model: BaselineModel,
    train_loader: DataLoader,
    val_loader: DataLoader,
    config: BaselineConfig,
    device: torch.device,
    output_dir: Path,
):
    """Main training loop for baseline model."""
    
    # Create output directory
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = output_dir / f"run_{timestamp}"
    run_dir.mkdir(parents=True, exist_ok=True)
    
    # Reconfigure logger with file handler
    global logger
    logger = setup_logging(run_dir)
    
    logger.info("=" * 70)
    logger.info("STARTING TRAINING")
    logger.info("=" * 70)
    logger.info(f"Output directory: {run_dir}")
    logger.info(f"Device: {device}")
    logger.info(f"Temporal model: {config.temporal_model}")
    logger.info(f"Epochs: {config.epochs}")
    logger.info(f"Batch size: {config.batch_size}")
    logger.info(f"Gradient accumulation: {config.grad_accum}")
    logger.info(f"Effective batch size: {config.batch_size * config.grad_accum}")
    logger.info(f"Learning rate: {config.learning_rate}")
    logger.info("=" * 70)
    
    # Optimizer
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=config.learning_rate,
        weight_decay=config.weight_decay,
    )
    
    # Scheduler
    steps_per_epoch = math.ceil(len(train_loader) / config.grad_accum)
    total_steps = config.epochs * steps_per_epoch
    warmup_steps = int(config.warmup_ratio * total_steps)
    
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    
    logger.info(f"Total steps: {total_steps}")
    logger.info(f"Warmup steps: {warmup_steps}")
    logger.info(f"Steps per epoch: {steps_per_epoch}")
    
    # Training state
    global_step = 0
    best_val_acc = 0.0
    epochs_without_improvement = 0
    training_history = []
    
    model.train()
    optimizer.zero_grad()
    
    start_time = time.time()
    
    # Training loop
    for epoch in range(config.epochs):
        logger.info("\n" + "=" * 70)
        logger.info(f"EPOCH {epoch + 1}/{config.epochs}")
        logger.info("=" * 70)
        
        epoch_loss = 0.0
        epoch_batches = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.epochs}")
        
        for batch_idx, batch in enumerate(pbar):
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)
            tasks = batch["tasks"]
            
            # Forward pass
            output = model(pixel_values, labels, tasks)
            loss = output["loss"] / config.grad_accum
            
            # Backward pass
            loss.backward()
            
            epoch_loss += loss.item() * config.grad_accum
            epoch_batches += 1
            
            # Optimizer step
            if (batch_idx + 1) % config.grad_accum == 0:
                # Gradient clipping
                torch.nn.utils.clip_grad_norm_(trainable_params, config.max_grad_norm)
                
                optimizer.step()
                optimizer.zero_grad()
                scheduler.step()
                global_step += 1
                
                # Logging
                if global_step % config.log_every_steps == 0:
                    lr = scheduler.get_last_lr()[0]
                    avg_loss = epoch_loss / epoch_batches
                    
                    pbar.set_postfix({
                        "loss": f"{avg_loss:.4f}",
                        "lr": f"{lr:.2e}",
                        "step": global_step,
                    })
                    
                    training_history.append({
                        "step": global_step,
                        "epoch": epoch + 1,
                        "loss": avg_loss,
                        "lr": lr,
                        "task_losses": output.get("task_losses", {}),
                    })
                
                # Validation
                if config.eval_every_steps > 0 and global_step % config.eval_every_steps == 0:
                    logger.info(f"\n{'='*70}")
                    logger.info(f"VALIDATION AT STEP {global_step}")
                    logger.info('='*70)
                    
                    val_metrics = evaluate_model(model, val_loader, device, "validation")
                    log_metrics(val_metrics, logger, global_step)
                    
                    # Check for improvement
                    val_acc = val_metrics["accuracy"]
                    if val_acc > best_val_acc + config.early_stopping_min_delta:
                        logger.info(f"\n✓ New best validation accuracy: {val_acc:.4f} (previous: {best_val_acc:.4f})")
                        best_val_acc = val_acc
                        epochs_without_improvement = 0
                        
                        # Save best checkpoint
                        save_checkpoint(
                            model, optimizer, scheduler, epoch, global_step,
                            val_metrics, run_dir, "checkpoint_best"
                        )
                    else:
                        epochs_without_improvement += 1
                        logger.info(f"No improvement for {epochs_without_improvement} evaluations")
                    
                    # Early stopping
                    if config.early_stopping and epochs_without_improvement >= config.early_stopping_patience:
                        logger.info(f"\n⚠ Early stopping triggered after {epochs_without_improvement} evaluations without improvement")
                        break
                    
                    model.train()
                
                # Save periodic checkpoint
                if config.save_every_steps > 0 and global_step % config.save_every_steps == 0:
                    save_checkpoint(
                        model, optimizer, scheduler, epoch, global_step,
                        {"step": global_step}, run_dir, f"checkpoint_step_{global_step}"
                    )
        
        # End of epoch
        avg_epoch_loss = epoch_loss / epoch_batches if epoch_batches > 0 else 0
        logger.info(f"\nEpoch {epoch+1} complete - Average loss: {avg_epoch_loss:.4f}")
        
        # Save epoch checkpoint
        save_checkpoint(
            model, optimizer, scheduler, epoch + 1, global_step,
            {"epoch": epoch + 1, "loss": avg_epoch_loss},
            run_dir, f"checkpoint_epoch_{epoch+1}"
        )
        
        # Early stopping check
        if config.early_stopping and epochs_without_improvement >= config.early_stopping_patience:
            logger.info("Early stopping triggered - ending training")
            break
    
    # Save final checkpoint
    logger.info("\n" + "=" * 70)
    logger.info("TRAINING COMPLETE - SAVING FINAL MODEL")
    logger.info("=" * 70)
    
    save_checkpoint(
        model, optimizer, scheduler, config.epochs, global_step,
        {"final": True}, run_dir, "checkpoint_final"
    )
    
    # Save training history
    history_file = run_dir / "training_history.json"
    with open(history_file, "w") as f:
        json.dump(training_history, f, indent=2)
    logger.info(f"Saved training history: {history_file}")
    
    # Final validation
    logger.info("\n" + "=" * 70)
    logger.info("FINAL VALIDATION")
    logger.info("=" * 70)
    
    val_metrics = evaluate_model(model, val_loader, device, "validation")
    log_metrics(val_metrics, logger)
    
    # Training summary
    total_time = time.time() - start_time
    logger.info("\n" + "=" * 70)
    logger.info("TRAINING SUMMARY")
    logger.info("=" * 70)
    logger.info(f"Total time: {format_time(total_time)}")
    logger.info(f"Total steps: {global_step}")
    logger.info(f"Best validation accuracy: {best_val_acc:.4f}")
    logger.info(f"Final validation accuracy: {val_metrics['accuracy']:.4f}")
    logger.info(f"Output directory: {run_dir}")
    logger.info("=" * 70)
    
    return run_dir, val_metrics


print("\n✓ Training loop defined")
print("Ready to train!")
print("\nTo start training, run:")
print("  run_dir, final_metrics = train_baseline_model(model, train_loader, val_loader, config, device, Path(config.output_dir))")


✓ Training loop defined
Ready to train!

To start training, run:
  run_dir, final_metrics = train_baseline_model(model, train_loader, val_loader, config, device, Path(config.output_dir))


In [10]:
# Cell 9: Start Training

from pathlib import Path

# Start training
logger.info("\n" + "=" * 70)
logger.info("INITIATING BASELINE MODEL TRAINING")
logger.info("=" * 70)

run_dir, final_metrics = train_baseline_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=config,
    device=device,
    output_dir=Path(config.output_dir),
)

print("\n" + "=" * 70)
print("TRAINING COMPLETED!")
print("=" * 70)
print(f"Output directory: {run_dir}")
print(f"Best validation accuracy: {final_metrics['accuracy']:.4f}")
print("=" * 70)

2026-01-18 01:42:24 | INFO     | 
2026-01-18 01:42:24 | INFO     | INITIATING BASELINE MODEL TRAINING
2026-01-18 01:42:24 | INFO     | ======================================================================
2026-01-18 01:42:24 | INFO     | ======================================================================
2026-01-18 01:42:24 | INFO     | STARTING TRAINING
2026-01-18 01:42:24 | INFO     | ======================================================================
2026-01-18 01:42:24 | INFO     | Output directory: /mnt/share/ali/VLM_Project/checkpoints_baseline/run_20260118_014224
2026-01-18 01:42:24 | INFO     | Device: cuda
2026-01-18 01:42:24 | INFO     | Temporal model: transformer
2026-01-18 01:42:24 | INFO     | Epochs: 20
2026-01-18 01:42:24 | INFO     | Batch size: 4
2026-01-18 01:42:24 | INFO     | Gradient accumulation: 4
2026-01-18 01:42:24 | INFO     | Effective batch size: 16
2026-01-18 01:42:24 | INFO     | Learning rate: 0.0001
2026-01-18 01:42:24 | INFO     | ==============


TRAINING COMPLETED!
Output directory: /mnt/share/ali/VLM_Project/checkpoints_baseline/run_20260118_014224
Best validation accuracy: 0.8152


In [11]:
# Cell 10: Final Test Evaluation Function

def final_test_evaluation(
    model: BaselineModel,
    test_loader: DataLoader,
    device: torch.device,
    checkpoint_path: Optional[Path] = None,
) -> Dict[str, Any]:
    """
    Run final evaluation on test set.
    Load best checkpoint if provided.
    """
    
    if checkpoint_path:
        logger.info("=" * 70)
        logger.info("LOADING BEST CHECKPOINT FOR TEST EVALUATION")
        logger.info("=" * 70)
        load_checkpoint(model, checkpoint_path)
        model = model.to(device)
    
    logger.info("=" * 70)
    logger.info("FINAL TEST EVALUATION")
    logger.info("=" * 70)
    
    test_metrics = evaluate_model(model, test_loader, device, "test")
    
    logger.info("\n" + "=" * 70)
    logger.info("TEST RESULTS")
    logger.info("=" * 70)
    log_metrics(test_metrics, logger)
    
    # Detailed per-task breakdown
    logger.info("\n" + "=" * 70)
    logger.info("DETAILED TEST RESULTS")
    logger.info("=" * 70)
    
    if "task_metrics" in test_metrics:
        for task, metrics in test_metrics["task_metrics"].items():
            logger.info(f"\n{task.upper()}:")
            logger.info(f"  Accuracy: {metrics['accuracy']*100:.2f}%")
            logger.info(f"  Correct: {metrics['correct']}/{metrics['total']}")
    
    logger.info("\n" + "=" * 70)
    logger.info(f"OVERALL TEST ACCURACY: {test_metrics['accuracy']*100:.2f}%")
    logger.info("=" * 70)
    
    return test_metrics


print("\n✓ Test evaluation function defined")
print("\nAfter training completes, run:")
print("  test_metrics = final_test_evaluation(model, test_loader, device, run_dir / 'checkpoint_best')")


✓ Test evaluation function defined

After training completes, run:
  test_metrics = final_test_evaluation(model, test_loader, device, run_dir / 'checkpoint_best')


In [14]:
test_metrics = final_test_evaluation(model, test_loader, device, run_dir / 'checkpoint_best')

2026-01-18 22:53:37 | INFO     | ======================================================================
2026-01-18 22:53:37 | INFO     | LOADING BEST CHECKPOINT FOR TEST EVALUATION
2026-01-18 22:53:37 | INFO     | ======================================================================
2026-01-18 22:53:37 | INFO     | Loading checkpoint: /mnt/share/ali/VLM_Project/checkpoints_baseline/run_20260118_014224/checkpoint_best/model.pt
2026-01-18 22:53:38 | INFO     | Loaded checkpoint from epoch 12, step 500
2026-01-18 22:53:38 | INFO     | ======================================================================
2026-01-18 22:53:38 | INFO     | FINAL TEST EVALUATION
2026-01-18 22:53:38 | INFO     | ======================================================================
2026-01-18 22:55:04 | INFO     |                                
2026-01-18 22:55:04 | INFO     | TEST RESULTS
2026-01-18 22:55:04 | INFO     | ======================================================================
2026-01-18 22:55

In [13]:
# Cell 11: Fix Checkpoint Loading

def load_checkpoint(model: BaselineModel, checkpoint_path: Path, optimizer=None, scheduler=None):
    """Load model from checkpoint."""
    ckpt_file = checkpoint_path / "model.pt"
    
    if not ckpt_file.exists():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_file}")
    
    logger.info(f"Loading checkpoint: {ckpt_file}")
    # Fix: Add weights_only=False since this is our own trusted checkpoint
    checkpoint = torch.load(ckpt_file, map_location="cpu", weights_only=False)
    
    model.load_state_dict(checkpoint["model_state_dict"])
    
    if optimizer is not None and "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    
    if scheduler is not None and "scheduler_state_dict" in checkpoint:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    
    epoch = checkpoint.get("epoch", 0)
    step = checkpoint.get("step", 0)
    metrics = checkpoint.get("metrics", {})
    
    logger.info(f"Loaded checkpoint from epoch {epoch}, step {step}")
    return epoch, step, metrics


print("✓ Fixed load_checkpoint function")
print("Now re-run the test evaluation:")
print("  test_metrics = final_test_evaluation(model, test_loader, device, run_dir / 'checkpoint_best')")

✓ Fixed load_checkpoint function
Now re-run the test evaluation:
  test_metrics = final_test_evaluation(model, test_loader, device, run_dir / 'checkpoint_best')


In [15]:
test_metrics = final_test_evaluation(model, test_loader, device, run_dir / 'checkpoint_best')


2026-01-19 01:36:53 | INFO     | ======================================================================
2026-01-19 01:36:53 | INFO     | LOADING BEST CHECKPOINT FOR TEST EVALUATION
2026-01-19 01:36:53 | INFO     | ======================================================================
2026-01-19 01:36:53 | INFO     | Loading checkpoint: /mnt/share/ali/VLM_Project/checkpoints_baseline/run_20260118_014224/checkpoint_best/model.pt
2026-01-19 01:36:55 | INFO     | Loaded checkpoint from epoch 12, step 500
2026-01-19 01:36:55 | INFO     | ======================================================================
2026-01-19 01:36:55 | INFO     | FINAL TEST EVALUATION
2026-01-19 01:36:55 | INFO     | ======================================================================
2026-01-19 01:39:38 | INFO     |                                
2026-01-19 01:39:38 | INFO     | TEST RESULTS
2026-01-19 01:39:38 | INFO     | ======================================================================
2026-01-19 01:39

In [16]:
# Cell 12: Enhanced Evaluation with F1, Kappa, and Per-Class Metrics

from sklearn.metrics import (
    f1_score, 
    cohen_kappa_score, 
    classification_report,
    confusion_matrix
)

@torch.no_grad()
def evaluate_model_detailed(
    model: BaselineModel,
    dataloader: DataLoader,
    device: torch.device,
    label_mappings: Dict[str, Dict[str, int]],
    split_name: str = "validation"
) -> Dict[str, Any]:
    """
    Evaluate model with detailed metrics including F1, Kappa, and per-class metrics.
    """
    model.eval()
    
    # Collect predictions by task
    task_predictions = defaultdict(list)
    task_labels = defaultdict(list)
    
    total_loss = 0.0
    num_batches = 0
    
    for batch in tqdm(dataloader, desc=f"Evaluating {split_name}", leave=False):
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)
        tasks = batch["tasks"]
        
        # Get predictions for each sample
        for i in range(len(tasks)):
            task = tasks[i]
            label = labels[i].item()
            
            # Skip invalid labels
            if label < 0:
                continue
            
            pred = model.predict(pixel_values[i:i+1], task)
            
            task_predictions[task].append(pred.item())
            task_labels[task].append(label)
        
        # Compute loss
        output = model(pixel_values, labels, tasks)
        total_loss += output["loss"].item()
        num_batches += 1
    
    # Compute metrics for each task
    results = {
        "loss": total_loss / num_batches if num_batches > 0 else 0.0,
        "split": split_name,
        "task_metrics": {},
    }
    
    all_predictions = []
    all_labels = []
    
    for task in task_predictions.keys():
        preds = np.array(task_predictions[task])
        labels = np.array(task_labels[task])
        
        all_predictions.extend(preds)
        all_labels.extend(labels)
        
        # Basic metrics
        correct = (preds == labels).sum()
        total = len(labels)
        accuracy = correct / total if total > 0 else 0.0
        
        # F1 scores
        macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
        weighted_f1 = f1_score(labels, preds, average='weighted', zero_division=0)
        
        # Cohen's Kappa
        kappa = cohen_kappa_score(labels, preds)
        
        # Per-class F1 scores
        per_class_f1 = f1_score(labels, preds, average=None, zero_division=0)
        
        # Get label names
        reverse_mapping = {idx: label for label, idx in label_mappings[task].items()}
        
        # Per-class metrics
        class_metrics = {}
        for class_idx, f1 in enumerate(per_class_f1):
            class_name = reverse_mapping.get(class_idx, f"class_{class_idx}")
            class_mask = labels == class_idx
            class_correct = (preds[class_mask] == labels[class_mask]).sum()
            class_total = class_mask.sum()
            class_acc = class_correct / class_total if class_total > 0 else 0.0
            
            class_metrics[class_name] = {
                "f1": float(f1),
                "accuracy": float(class_acc),
                "support": int(class_total),
                "correct": int(class_correct),
            }
        
        # Confusion matrix
        conf_matrix = confusion_matrix(labels, preds)
        
        results["task_metrics"][task] = {
            "accuracy": float(accuracy),
            "correct": int(correct),
            "total": int(total),
            "macro_f1": float(macro_f1),
            "weighted_f1": float(weighted_f1),
            "kappa": float(kappa),
            "per_class_metrics": class_metrics,
            "confusion_matrix": conf_matrix.tolist(),
        }
    
    # Overall metrics (across all tasks)
    all_predictions = np.array(all_predictions)
    all_labels = np.array(all_labels)
    
    overall_accuracy = (all_predictions == all_labels).sum() / len(all_labels) if len(all_labels) > 0 else 0.0
    overall_macro_f1 = f1_score(all_labels, all_predictions, average='macro', zero_division=0)
    overall_weighted_f1 = f1_score(all_labels, all_predictions, average='weighted', zero_division=0)
    overall_kappa = cohen_kappa_score(all_labels, all_predictions)
    
    results["overall"] = {
        "accuracy": float(overall_accuracy),
        "macro_f1": float(overall_macro_f1),
        "weighted_f1": float(overall_weighted_f1),
        "kappa": float(overall_kappa),
        "total": len(all_labels),
        "correct": int((all_predictions == all_labels).sum()),
    }
    
    return results


def log_detailed_metrics(results: Dict[str, Any], logger):
    """Log detailed evaluation metrics in a readable format."""
    
    split = results.get("split", "eval")
    
    logger.info("=" * 70)
    logger.info(f"{split.upper()} - DETAILED METRICS")
    logger.info("=" * 70)
    
    # Overall metrics
    if "overall" in results:
        overall = results["overall"]
        logger.info("\nOVERALL METRICS:")
        logger.info(f"  Accuracy:    {overall['accuracy']*100:.2f}% ({overall['correct']}/{overall['total']})")
        logger.info(f"  Macro F1:    {overall['macro_f1']:.4f}")
        logger.info(f"  Weighted F1: {overall['weighted_f1']:.4f}")
        logger.info(f"  Cohen's κ:   {overall['kappa']:.4f}")
    
    # Per-task metrics
    if "task_metrics" in results:
        for task, metrics in results["task_metrics"].items():
            logger.info("\n" + "-" * 70)
            logger.info(f"TASK: {task.upper()}")
            logger.info("-" * 70)
            logger.info(f"  Accuracy:    {metrics['accuracy']*100:.2f}% ({metrics['correct']}/{metrics['total']})")
            logger.info(f"  Macro F1:    {metrics['macro_f1']:.4f}")
            logger.info(f"  Weighted F1: {metrics['weighted_f1']:.4f}")
            logger.info(f"  Cohen's κ:   {metrics['kappa']:.4f}")
            
            # Per-class metrics
            logger.info("\n  Per-Class Metrics:")
            logger.info(f"  {'Class':<50s} {'F1':>8s} {'Acc':>8s} {'Support':>10s}")
            logger.info("  " + "-" * 78)
            
            for class_name, class_metrics in sorted(
                metrics["per_class_metrics"].items(), 
                key=lambda x: x[1]["f1"], 
                reverse=True
            ):
                f1 = class_metrics["f1"]
                acc = class_metrics["accuracy"]
                support = class_metrics["support"]
                
                # Truncate long class names
                display_name = class_name[:48] + ".." if len(class_name) > 50 else class_name
                
                logger.info(f"  {display_name:<50s} {f1:>8.4f} {acc*100:>7.2f}% {support:>10d}")
    
    logger.info("=" * 70)


def save_detailed_results(results: Dict[str, Any], output_path: Path):
    """Save detailed results to JSON file."""
    
    # Convert numpy arrays to lists for JSON serialization
    def convert_numpy(obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, dict):
            return {k: convert_numpy(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [convert_numpy(item) for item in obj]
        else:
            return obj
    
    results_serializable = convert_numpy(results)
    
    with open(output_path, "w") as f:
        json.dump(results_serializable, f, indent=2)
    
    logger.info(f"Saved detailed results to: {output_path}")


def final_test_evaluation_detailed(
    model: BaselineModel,
    test_loader: DataLoader,
    label_mappings: Dict[str, Dict[str, int]],
    device: torch.device,
    checkpoint_path: Optional[Path] = None,
    output_dir: Optional[Path] = None,
) -> Dict[str, Any]:
    """
    Run detailed final evaluation on test set with all metrics.
    """
    
    if checkpoint_path:
        logger.info("=" * 70)
        logger.info("LOADING BEST CHECKPOINT FOR TEST EVALUATION")
        logger.info("=" * 70)
        load_checkpoint(model, checkpoint_path)
        model = model.to(device)
    
    logger.info("\n" + "=" * 70)
    logger.info("FINAL TEST EVALUATION (DETAILED)")
    logger.info("=" * 70)
    
    # Run detailed evaluation
    test_results = evaluate_model_detailed(
        model=model,
        dataloader=test_loader,
        device=device,
        label_mappings=label_mappings,
        split_name="test"
    )
    
    # Log detailed metrics
    log_detailed_metrics(test_results, logger)
    
    # Save results
    if output_dir:
        results_path = output_dir / "test_results_detailed.json"
        save_detailed_results(test_results, results_path)
    
    return test_results


print("\n✓ Enhanced evaluation functions defined")
print("\nAvailable functions:")
print("  - evaluate_model_detailed(): Full metrics including F1, Kappa, per-class")
print("  - log_detailed_metrics(): Pretty print all metrics")
print("  - save_detailed_results(): Save to JSON")
print("  - final_test_evaluation_detailed(): Complete test evaluation")
print("\nTo run detailed test evaluation:")
print("  test_results = final_test_evaluation_detailed(")
print("      model, test_loader, label_mappings, device,")
print("      checkpoint_path=run_dir / 'checkpoint_best',")
print("      output_dir=run_dir")
print("  )")


✓ Enhanced evaluation functions defined

Available functions:
  - evaluate_model_detailed(): Full metrics including F1, Kappa, per-class
  - log_detailed_metrics(): Pretty print all metrics
  - save_detailed_results(): Save to JSON
  - final_test_evaluation_detailed(): Complete test evaluation

To run detailed test evaluation:
  test_results = final_test_evaluation_detailed(
      model, test_loader, label_mappings, device,
      checkpoint_path=run_dir / 'checkpoint_best',
      output_dir=run_dir
  )


In [18]:
final_test_evaluation_detailed()

TypeError: final_test_evaluation_detailed() missing 4 required positional arguments: 'model', 'test_loader', 'label_mappings', and 'device'